In [26]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, avg,udf
# Initialize
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()
# Create Base DataFrame
data = [
    (1, "Alice", "Engineering", 75000, 25),
    (2, "Bob", "Marketing", 60000, 30),
    (3, "Charlie", "Engineering", 80000, 35),
    (4, "David", "Sales", 65000, 28),
    (5, "Eve", "Marketing", 70000, 32)
]
columns = ["Id", "Name", "Department", "Salary", "Age"]
df = spark.createDataFrame(data, columns);

In [2]:
df_select = df.select("Name","Department","Salary")

In [3]:
df_select.show()

+-------+-----------+------+
|   Name| Department|Salary|
+-------+-----------+------+
|  Alice|Engineering| 75000|
|    Bob|  Marketing| 60000|
|Charlie|Engineering| 80000|
|  David|      Sales| 65000|
|    Eve|  Marketing| 70000|
+-------+-----------+------+



In [4]:
df_filter = df.filter(col("salary")>65000)

In [5]:
df_filter.show()

+---+-------+-----------+------+---+
| Id|   Name| Department|Salary|Age|
+---+-------+-----------+------+---+
|  1|  Alice|Engineering| 75000| 25|
|  3|Charlie|Engineering| 80000| 35|
|  5|    Eve|  Marketing| 70000| 32|
+---+-------+-----------+------+---+



In [6]:
df_with_col = df.withColumn("Bonus",col("Salary")*0.10)
df_with_col.show()

+---+-------+-----------+------+---+------+
| Id|   Name| Department|Salary|Age| Bonus|
+---+-------+-----------+------+---+------+
|  1|  Alice|Engineering| 75000| 25|7500.0|
|  2|    Bob|  Marketing| 60000| 30|6000.0|
|  3|Charlie|Engineering| 80000| 35|8000.0|
|  4|  David|      Sales| 65000| 28|6500.0|
|  5|    Eve|  Marketing| 70000| 32|7000.0|
+---+-------+-----------+------+---+------+



In [7]:
df.write.mode("overwrite").csv("output/employeetable",header = True)

In [8]:
df.toPandas().to_csv(
    "output/employee123.csv",
    index = False
)

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, avg
# Initialize
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()
# Create Base DataFrame
data = [
    (101, "Aarav", "HR", 45000),
    (102, "Diya", "IT", 60000),
    (103, "Vivaan", "Finance", 55000),
    (104, "Anaya", "Marketing", 50000),
    (105, "Aditya", "IT", 65000),
    (106, "Ishita", "HR", 47000),
    (107, "Krishna", "Sales", 52000),
    (108, "Meera", "Finance", 58000),
    (109, "Arjun", "IT", 70000),
    (110, "Riya", "Marketing", 49000),
    (111, "Kabir", "Sales", 54000),
    (112, "Saanvi", "HR", 46000),
    (113, "Reyansh", "IT", 72000),
    (114, "Aadhya", "Finance", 61000),
    (115, "Atharv", "Sales", 53000),
    (116, "Myra", "Marketing", 51000),
    (117, "Shaurya", "IT", 68000),
    (118, "Pari", "HR", 48000),
    (119, "Yash", "Finance", 59000),
    (120, "Kiara", "Sales", 56000)
]
columns = ["Id", "Name", "Department", "Salary"]
df = spark.createDataFrame(data, columns);

In [10]:
df.write.mode("overwrite").csv("output/employee_data",header = True)

In [11]:
df.toPandas().to_csv(
    "output/employeetable2.csv",
    index = False
)

In [12]:
df.take(10)

[Row(Id=101, Name='Aarav', Department='HR', Salary=45000),
 Row(Id=102, Name='Diya', Department='IT', Salary=60000),
 Row(Id=103, Name='Vivaan', Department='Finance', Salary=55000),
 Row(Id=104, Name='Anaya', Department='Marketing', Salary=50000),
 Row(Id=105, Name='Aditya', Department='IT', Salary=65000),
 Row(Id=106, Name='Ishita', Department='HR', Salary=47000),
 Row(Id=107, Name='Krishna', Department='Sales', Salary=52000),
 Row(Id=108, Name='Meera', Department='Finance', Salary=58000),
 Row(Id=109, Name='Arjun', Department='IT', Salary=70000),
 Row(Id=110, Name='Riya', Department='Marketing', Salary=49000)]

In [13]:
print("before partitionings:",df.rdd.getNumPartitions())

before partitionings: 12


In [14]:
df_repartition = df.repartition(5)
print("After partition:",df_repartition.rdd.getNumPartitions())

[Stage 16:>                                                       (0 + 12) / 12]

After partition: 5


In [15]:
df_colesced = df_repartition.coalesce(2)
print("after coalesced:",df_colesced.rdd.getNumPartitions())

[Stage 17:=======================>                                 (5 + 7) / 12]

after coalesced: 2


In [16]:
df_repartition.write.mode("overwrite").csv("output/employee_data_afterP",header = True)

In [17]:
df_colesced.write.mode("overwrite").csv("output/employee_data_colesce",header = True)

In [18]:
optimized_df = df.filter(col("Salary")>55000).filter(col("Salary")<70000)
optimized_df.show()

+---+-------+----------+------+
| Id|   Name|Department|Salary|
+---+-------+----------+------+
|102|   Diya|        IT| 60000|
|105| Aditya|        IT| 65000|
|108|  Meera|   Finance| 58000|
|114| Aadhya|   Finance| 61000|
|117|Shaurya|        IT| 68000|
|119|   Yash|   Finance| 59000|
|120|  Kiara|     Sales| 56000|
+---+-------+----------+------+



In [19]:
optimized_df.explain()

== Physical Plan ==
*(1) Filter (isnotnull(Salary#87L) AND ((Salary#87L > 55000) AND (Salary#87L < 70000)))
+- *(1) Scan ExistingRDD[Id#84L,Name#85,Department#86,Salary#87L]




In [20]:
import time
start_time = time.time()
count_first_cache = optimized_df.count()
end_time = time.time()
print(f"1. Optimized execution | count: {count_first_cache} | time: {round(end_time - start_time, 2)} seconds")

1. Optimized execution | count: 7 | time: 1.2 seconds


In [21]:
optimized_df.cache()

DataFrame[Id: bigint, Name: string, Department: string, Salary: bigint]

In [22]:
start_time = time.time()
count_uncached = optimized_df.count()
end_time = time.time()
print(f"1. Optimized execution | count: {count_uncached} | time: {round(end_time - start_time, 4)} seconds")

[Stage 30:=======================>                                 (5 + 7) / 12]

1. Optimized execution | count: 7 | time: 2.5702 seconds


In [34]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

# Sample Data
data = [
    ("Alice", 25),
    ("Bob", 17),
    ("Charlie", 35),
    ("David", 15)
]

df = spark.createDataFrame(data, ["Name", "Age"])

df.show()

+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 17|
|Charlie| 35|
|  David| 15|
+-------+---+



In [36]:

def categorize_age(age):
    if age >= 18:
        return "Adult"
    return "Minor"


In [37]:
age_category_udf = udf(categorize_age, StringType())

In [38]:
df_method1 = df.withColumn(
    "Category",
    age_category_udf(col("Age"))
)


In [39]:
print("Method 1: Standard UDF via DataFrame API")
df_method1.show()

Method 1: Standard UDF via DataFrame API


[Stage 42:=================================================>        (6 + 1) / 7]

+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   Adult|
|    Bob| 17|   Minor|
|Charlie| 35|   Adult|
|  David| 15|   Minor|
+-------+---+--------+



In [40]:
from pyspark.sql.types import StringType

spark.udf.register(
    "sql_categorize_age",
    categorize_age,
    StringType()
)

26/06/13 06:58:42 WARN SimpleFunctionRegistry: The function sql_categorize_age replaced a previously registered function.


<function __main__.categorize_age(age)>

In [41]:
df.createOrReplaceTempView("people")

In [42]:
sql_df = spark.sql("""
SELECT Name,
       Age,
       sql_categorize_age(Age) AS Category
FROM people
""")

sql_df.show()

[Stage 45:=================================================>        (6 + 1) / 7]

+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   Adult|
|    Bob| 17|   Minor|
|Charlie| 35|   Adult|
|  David| 15|   Minor|
+-------+---+--------+

